In [1]:
import os
import pandas as pd

norm = './data/Training_Data/normal/'
fault = './data/Training_Data/fault/'

# 获取所有名称（文件 + 文件夹）
norms = os.listdir(norm)
faults = os.listdir(fault)

df_norms = pd.DataFrame()
for i in norms:
    df_norms_i = pd.read_excel(norm+i)
    df_norms = pd.concat([df_norms,df_norms_i])
df_faults = pd.DataFrame()
for i in faults:
    df_faults_i = pd.read_excel(fault+i)
    df_faults = pd.concat([df_faults,df_faults_i])
    
df_train = pd.concat([df_norms,df_faults]).reset_index(drop=True)
del df_norms,df_norms_i,df_faults,df_faults_i

In [2]:
df_norms = pd.DataFrame()
for i in norms:
    df_norms_i = pd.read_excel(norm+i)
    df_norms = pd.concat([df_norms,df_norms_i])
df_faults = pd.DataFrame()
for i in faults:
    df_faults_i = pd.read_excel(fault+i)
    df_faults = pd.concat([df_faults,df_faults_i])

In [3]:
test_df = pd.read_excel(r'./data/test_data_all.xlsx')

In [4]:
set(df_train.columns)-set(test_df.columns)

{'Class', 'Heat Balance'}

In [5]:
set(test_df.columns)-set(df_train.columns)

{'ID'}

In [6]:
df_train = df_train.drop('Heat Balance',axis=1)

In [7]:
df_train['Class'].value_counts()

Class
normal    4844
cf1         75
cf2         75
cf3         75
cf4         75
eo1         75
eo2         75
eo3         75
eo4         75
fwc1        75
fwc2        75
fwc3        75
fwc4        75
fwe1        75
fwe2        75
fwe3        75
fwe4        75
nc1         75
nc2         75
nc3         75
nc4         75
rl1         75
rl2         75
rl3         75
rl4         75
ro1         75
ro2         75
ro3         75
ro4         75
Name: count, dtype: int64

# 数据处理

In [8]:
# 基础数据处理库
import pandas as pd
import numpy as np
from scipy.stats import randint as sp_randint, uniform

# 模型相关库
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb

# 数据预处理与特征工程
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold

# 模型评估与交叉验证
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score

# 警告抑制
import warnings
warnings.filterwarnings('ignore')  # 全局抑制所有非致命警告

In [9]:
x_train = df_train.drop(['Class'], axis=1)
y_train = df_train['Class']
x_test = test_df.drop(['ID'], axis=1)

# 預處理流程 
median_vals = x_train.median()
x_train = x_train.fillna(median_vals)
x_test = x_test.fillna(median_vals)

# 方差阈值法（剔除低方差无区分度特征）
vt = VarianceThreshold(threshold=0.01)
x_train = vt.fit_transform(x_train)
x_test = vt.transform(x_test)

# 標準化/歸一化
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# 標籤編碼
le = LabelEncoder()
y_train = le.fit_transform(y_train)

# 多跑一些模型，看看哪个好些

In [10]:
# 模型配置
num_classes = len(le.classes_)
models = {
    'XGBoost': xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=num_classes,
        max_depth=4,
        learning_rate=0.03,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        verbosity=0
    ),
    'RF': RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        max_features='sqrt',
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ),
    'LGBM': lgb.LGBMClassifier(
        boosting_type='gbdt',
        num_leaves=31,
        learning_rate=0.05,
        n_estimators=150,
        objective='multiclass',
        num_class=num_classes,
        class_weight='balanced',  # 可选，平衡类别
        random_state=42,
        importance_type='gain',
        verbose=-1,
        n_jobs=-1
    ),
    'GB': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        subsample=0.8,
        max_features='sqrt',
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
}

In [11]:
# 交叉驗證
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = {f'{name}_{avg}': [] for name in models.keys() for avg in ['Macro', 'Micro']}

for train_idx, val_idx in skf.split(x_train, y_train):
    # numpy.ndarray 直接索引
    x_cv_train, x_cv_val = x_train[train_idx], x_train[val_idx]
    y_cv_train, y_cv_val = y_train[train_idx], y_train[val_idx]
    
    for name, model in models.items():
        # 重新创建模型实例，避免 fold 间状态泄露
        model = model.__class__(**model.get_params())
        model.fit(x_cv_train, y_cv_train)
        y_pred = model.predict(x_cv_val)
        # 存储分数
        fold_scores[f'{name}_Macro'].append(f1_score(y_cv_val, y_pred, average='macro'))
        fold_scores[f'{name}_Micro'].append(f1_score(y_cv_val, y_pred, average='micro'))

# 結果輸出
print("5折交叉驗證結果：")
# 構建結果列表
results = [
    {
        '模型': name,
        'Macro_F1': round(np.mean(fold_scores[f'{name}_Macro']), 4),
        'Micro_F1': round(np.mean(fold_scores[f'{name}_Micro']), 4)
    }
    for name in models.keys()
]
# 格式化列印
print(f"{'模型':<12}{'Macro_F1':<10}{'Micro_F1':<10}\n{'-'*32}")
for r in results: print(f"{r['模型']:<12}{r['Macro_F1']:.4f}{' ':<6}{r['Micro_F1']:.4f}")

5折交叉驗證結果：
模型          Macro_F1  Micro_F1  
--------------------------------
XGBoost     0.7037      0.8923
RF          0.4614      0.2049
LGBM        0.7948      0.9183
GB          0.6574      0.8767
ExtraTrees  0.4501      0.2630


# 用lgbm、调参

In [12]:
from scipy.stats import randint as sp_randint,randint, uniform
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [13]:
x_train1, x_test1, y_train1, y_test1 = train_test_split(x_train, y_train, stratify=y_train, test_size=0.1, random_state=42)

In [14]:
# 定义超参数搜索空间
param_dist = {
    'learning_rate': uniform(0.01, 0.14),
    'max_depth': sp_randint(3, 15),
    'num_leaves': sp_randint(20, 200),
    'subsample': uniform(0.6, 0.4),       # 0.6~1.0
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),           # 0~1
    'reg_lambda': uniform(0, 1)
}

In [ ]:
N_ITER_SEARCH = 15   # 随机搜索尝试 15 组参数
N_FOLDS = 8          # 8 折交叉验证
EARLY_STOPPING_ROUNDS = 50 # 早停轮数
num_classes = len(le.classes_)

: 

In [ ]:
results = []
for i in range(N_ITER_SEARCH):
    print(f"\n--- 第 {i+1}/{N_ITER_SEARCH} 次搜索 ---")
    
    # --- 步骤 A: 随机生成一组参数 ---
    # 使用 .rvs() 方法从定义的分布中抽取一个样本
    # current_params = {k: v.rvs() for k, v in param_dist.items()}
    current_params = {}
    for key, value in param_dist.items():
        if isinstance(value, list):
            current_params[key] = np.random.choice(value)
        else:
            current_params[key] = value.rvs()
    
    # 补充模型固定参数 (不参与搜索的参数)
    current_params.update({
        'n_estimators':5000,
        'boosting_type':'gbdt',
        'objective':'multiclass',
        'num_class':num_classes,
        'class_weight':'balanced',  # 可选，平衡类别
        'random_state':42,
        'importance_type':'gain',
        'verbose':-1,
        'n_jobs':-1
    })
    
    cv_scores = []
    cv_iterations = []
    cv_macro = []
    cv_micro = []
    
    # --- 步骤 B: 交叉验证 ---
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(x_train1, y_train1)):
        # 划分当前折的训练集和验证集
        X_fold_train, X_fold_val = x_train1[train_idx], x_train1[val_idx]
        y_fold_train, y_fold_val = y_train1[train_idx], y_train1[val_idx]
        
        # 初始化模型 (使用当前随机参数)
        model = lgb.LGBMClassifier(**current_params)
        
        # 训练模型，传入 eval_set 启用早停
        model.fit(
            X_fold_train, y_fold_train,
            eval_set=[(X_fold_val, y_fold_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)
            ]
        )
        
        # --- 获取最佳分数和最佳轮数 ---
        # 我们需要提取具体的数值
        y_pred = model.predict(X_fold_val)
        # 存储分数
        best_score = model.best_score_['valid_0']['multi_logloss'] # auc
        macro_i = f1_score(y_fold_val, y_pred, average='macro')
        micro_i = f1_score(y_fold_val, y_pred, average='micro')
        best_iter = model.best_iteration_
        
        cv_scores.append(best_score)
        cv_macro.append(macro_i)
        cv_micro.append(micro_i) 
        cv_iterations.append(best_iter)
        
        # print(f"   Fold {fold_idx+1}: AUC={best_score:.5f}, Best Iter={best_iter}")
    
    # --- 步骤 C: 记录本次搜索的平均表现 ---
    mean_multi_logloss = np.mean(cv_scores)
    mean_macrof1 = np.mean(cv_macro)
    mean_microf1 = np.mean(cv_micro)
    mean_iter = int(np.mean(cv_iterations))
    
    print(f"参数: {current_params}")
    print(f"平均 macrof1: {mean_macrof1:.5f} | 平均 microf1:: {mean_microf1}")
    print(f"平均 mean_multi_logloss: {mean_multi_logloss:.5f} | 平均最佳轮数: {mean_iter}")
    
    results.append({
        'mean_score': mean_multi_logloss,
        'mean_macrof1':mean_macrof1,
        'mean_microf1':mean_microf1,
        'best_iter': mean_iter,
        'params': current_params
    })


--- 第 1/15 次搜索 ---
参数: {'learning_rate': np.float64(0.03841845003480424), 'max_depth': 6, 'num_leaves': 161, 'subsample': np.float64(0.7937299780311006), 'colsample_bytree': np.float64(0.7744427968158364), 'reg_alpha': np.float64(0.9357523429537329), 'reg_lambda': np.float64(0.7497732931406146), 'n_estimators': 5000, 'boosting_type': 'gbdt', 'objective': 'multiclass', 'num_class': 29, 'class_weight': 'balanced', 'random_state': 42, 'importance_type': 'gain', 'verbose': -1, 'n_jobs': -1}
平均 macrof1: 0.78804 | 平均 microf1:: 0.8987017021917602
平均 mean_multi_logloss: 0.36663 | 平均最佳轮数: 452

--- 第 2/15 次搜索 ---
参数: {'learning_rate': np.float64(0.09988040195396124), 'max_depth': 6, 'num_leaves': 127, 'subsample': np.float64(0.727096224248513), 'colsample_bytree': np.float64(0.7071681775690044), 'reg_alpha': np.float64(0.5457545656340227), 'reg_lambda': np.float64(0.318823691378911), 'n_estimators': 5000, 'boosting_type': 'gbdt', 'objective': 'multiclass', 'num_class': 29, 'class_weight': 'bala

In [ ]:
best_result = max(results, key=lambda x: x['mean_macrof1'])
best_params = best_result['params']
best_n_estimators = best_result['best_iter']
print(f"最佳 macrof1: {best_result['mean_macrof1']:.5f}")
print(f"最佳迭代轮数: {best_n_estimators}")
print(f"最佳参数: {best_params}")

最佳 macrof1: 0.81060
最佳迭代轮数: 652
最佳参数: {'learning_rate': np.float64(0.07102881273665658), 'max_depth': 4, 'num_leaves': 81, 'subsample': np.float64(0.974211468191233), 'colsample_bytree': np.float64(0.7085617508033017), 'reg_alpha': np.float64(0.2868295897483325), 'reg_lambda': np.float64(0.8606413014725919), 'n_estimators': 5000, 'boosting_type': 'gbdt', 'objective': 'multiclass', 'num_class': 29, 'class_weight': 'balanced', 'random_state': 42, 'importance_type': 'gain', 'verbose': -1, 'n_jobs': -1}


In [ ]:
print("\n5. 使用全量数据重新训练最终模型...")
final_params = best_params.copy()
final_params['n_estimators'] = best_n_estimators+20
# 重新训练时可以关闭 verbose 或者设为 100 看进度
# final_params['verbose'] = 100 

final_model = lgb.LGBMClassifier(**final_params)

# 使用所有的 X_train 进行训练 (不再需要 eval_set)
final_model.fit(x_train, y_train)


5. 使用全量数据重新训练最终模型...


LGBMClassifier(class_weight='balanced',
               colsample_bytree=np.float64(0.7085617508033017),
               importance_type='gain',
               learning_rate=np.float64(0.07102881273665658), max_depth=4,
               n_estimators=672, n_jobs=-1, num_class=29, num_leaves=81,
               objective='multiclass', random_state=42,
               reg_alpha=np.float64(0.2868295897483325),
               reg_lambda=np.float64(0.8606413014725919),
               subsample=np.float64(0.974211468191233), verbose=-1)

In [ ]:
# 在测试集上进行预测
y_test_pre = model.predict(x_test)
# 将预测结果还原为原始标签
y_test_pre = le.inverse_transform(y_test_pre)

result = test_df[['ID']]
result['Class'] = y_test_pre

In [ ]:
result

,ID,Class
0,1,normal
1,2,eo2
2,3,normal
3,4,normal
4,5,fwc4
...,...,...
3649,3650,nc1
3650,3651,normal
3651,3652,normal
3652,3653,ro4
